In [3]:
import os
import traceback
import pandas as pd
import xlwings as xw

from process_engine import (
    f1_confirm_test_file_folder,
    f3a_prompt_user_to_pick_file_or_folder,
    f4_2_1_detect_comm_type_label,
    f4_2_2_prompt_comm_type_only_if_needed,
    get_reader_for_comm_type,
    f4_4_run_engineered_reader,
    f4_5_show_processing_error_widget
)


# ---------------------------------------------------------------------------
# Main Function: process_comms_main
# ---------------------------------------------------------------------------
def process_comms_main():
    """
    Overall Process Commissions Main Function

    Step 1:
        Confirm whether this is:
        - Test run or Official run
        - File or Folder import
        - Whether to clear table

    Step 2:
        Get workbook, output sheet, comm_month, and lookup df

    Step 3:
        Get file path or folder path

    Step 4:
        Process file / each file in folder and return engineered df(s)

    Step 5:
        To be confirmed
    """

    try:
        # -------------------------------------------------------------------
        # Step 1: Confirm Test / Official / File / Folder / Clear Table
        # -------------------------------------------------------------------
        print("🟦 Step 1: Confirm Test / Official / File / Folder")
        test_official, file_folder, clear_table = f1_confirm_test_file_folder()
        print("✅ Step 1 complete")
        print(f"   [F1] test_official : {test_official}")
        print(f"   [F1] file_folder   : {file_folder}")
        print(f"   [F1] clear_table   : {clear_table}")

        # -------------------------------------------------------------------
        # Step 2: Get workbook, output sheet, comm_month, and comm tables df
        # -------------------------------------------------------------------
        print("🟦 Step 2: Get workbook, output sheet, comm_month, and comm tables df")
        current_wb, sh_output, comm_month, comm_tables_main_df = f2_get_core_variables()
        print("✅ Step 2 complete")
        print(f"   [F2] current_wb           : {current_wb.name if current_wb else 'None'}")
        print(f"   [F2] sh_output            : {sh_output.name if sh_output else 'None'}")
        print(f"   [F2] comm_month           : {comm_month}")
        print(f"   [F2] comm_tables_main_df  : {comm_tables_main_df.shape if isinstance(comm_tables_main_df, pd.DataFrame) else 'None'}")

        # ---------------------------------------------------------------------------
        # Step 3: Prompt user to pick file or folder
        # ---------------------------------------------------------------------------
        print("🟦 Step 3: Prompt user to pick file or folder")
        selected_path = f3a_prompt_user_to_pick_file_or_folder(file_folder)

        print("✅ Step 3 complete")
        print(f"   [Step 3] selected_path: {selected_path}")

        if not selected_path:
            print("🔴 No file/folder selected. Exiting.")
            return

        # -------------------------------------------------------------------
        # Step 4: Process file / each file in folder and apply DF
        # -------------------------------------------------------------------
        print("🟦 Step 4: Process file / each file in folder and apply DF")
        processed_results = f4_process_selected_path(
            selected_path=selected_path,
            file_folder=file_folder,
            comm_month=comm_month,
            comm_tables_main_df=comm_tables_main_df,
            sh_output=sh_output
        )
        print("✅ Step 4 complete")

        if processed_results is None:
            print("🔴 Step 4 returned None. Exiting.")
            return

        print("🟩 Step 4 processed results summary:")
        if isinstance(processed_results, pd.DataFrame):
            print(processed_results)
        else:
            print(f"   [F4] processed_results type: {type(processed_results)}")

        # -------------------------------------------------------------------
        # Step 5: To Be Confirmed
        # -------------------------------------------------------------------
        print("🟦 Step 5: To be confirmed")
        print("✅ Process completed up to current designed logic.")

    except Exception as e:
        print("🔴 ERROR in process_comms_main")
        print(f"   Error: {e}")
        print(traceback.format_exc())
        
        


# ---------------------------------------------------------------------------
# F2: Get workbook, output sheet, comm_month, and comm tables df
# ---------------------------------------------------------------------------
def f2_get_core_variables():
    """
    Returns:
      current_wb, sh_output, comm_month, comm_tables_main_df
    """
    print("   [F2] Getting active workbook...")
    current_wb = xw.books.active
    print(f"   [F2] current_wb: {current_wb.name if current_wb else 'None'}")

    # -----------------------------------------------------------------------
    # Output sheet
    # -----------------------------------------------------------------------
    print("   [F2] Checking output sheet...")
    out_name = "processed_comms_test"

    existing_sheet_names = [s.name.lower() for s in current_wb.sheets]
    print(f"   [F2] existing_sheet_names: {existing_sheet_names}")

    if out_name.lower() in existing_sheet_names:
        sh_output = current_wb.sheets[out_name]
        print(f"   [F2] Using existing output sheet: {sh_output.name}")
    else:
        sh_output = current_wb.sheets.add(out_name)
        print(f"   [F2] Created new output sheet: {sh_output.name}")

    # -----------------------------------------------------------------------
    # Comm month
    # -----------------------------------------------------------------------
    print("   [F2] Reading comm_month from Dashboard!C3...")
    comm_month = None

    try:
        sh_dash = current_wb.sheets["Dashboard"]
        comm_month = sh_dash.range("C3").value
        print("   [F2] ✅ Comm Month Read")
        print(f"   [F2] comm_month: {comm_month}")
    except Exception as e:
        comm_month = None
        print(f"   [F2] 🔴 Failed to read comm_month from Dashboard!C3: {e}")

    # -----------------------------------------------------------------------
    # Lookup df
    # -----------------------------------------------------------------------
    print("   [F2] Reading lookup table from Lookup_Tables...")
    comm_tables_main_df = pd.DataFrame()

    try:
        sh_lookup = current_wb.sheets["Lookup_Tables"]
        print("   [F2] Lookup_Tables sheet found")

        last_row = sh_lookup.range("BC4000").end("up").row
        print(f"   [F2] Raw last_row from BC4000.end('up'): {last_row}")

        if last_row < 3:
            last_row = 3
            print("   [F2] last_row adjusted to 3 because table is empty / minimal")

        lookup_range = f"BC3:BM{last_row}"
        print(f"   [F2] lookup_range: {lookup_range}")

        values = sh_lookup.range(lookup_range).value
        print(f"   [F2] Raw lookup values type: {type(values)}")

        expected_headers = [
            "Product House",
            "Contract Number Action", "Contract Number Details",
            "Client Name Action", "Client Name Details",
            "Planner Action", "Planner Details",
            "Total Commission Action", "Total Commission Details",
            "Commission Date Action", "Commission Date Details",
        ]

        comm_tables_main_df = pd.DataFrame(values, columns=expected_headers)
        comm_tables_main_df = comm_tables_main_df.dropna(how="all")

        print(f"   [F2] ✅ comm_tables_main_df shape: {comm_tables_main_df.shape}")
        print(f"   [F2] comm_tables_main_df columns: {list(comm_tables_main_df.columns)}")

    except Exception as e:
        comm_tables_main_df = pd.DataFrame()
        print(f"   [F2] 🔴 Failed to read lookup table: {e}")

    return current_wb, sh_output, comm_month, comm_tables_main_df


# ---------------------------------------------------------------------------
# F4: Process selected path (single file or folder of files)
# ---------------------------------------------------------------------------
def f4_process_selected_path(selected_path, file_folder, comm_month, comm_tables_main_df, sh_output):
    print("   [F4] Starting Step 4 processing...")

    file_folder_clean = str(file_folder).strip().lower()

    if file_folder_clean == "folder":
        print("   [F4] Folder mode detected.")
        excel_files = f4_1_get_excel_files_in_folder(selected_path)

        if not excel_files:
            print("🔴 No Excel files found in selected folder.")
            return pd.DataFrame([{
                "file_path": selected_path,
                "status": "error",
                "comm_type": "",
                "message": "No Excel files found in folder"
            }])

        processed_log = []

        for i, file_path in enumerate(excel_files, start=1):
            print("------------------------------------------------------------------")
            print(f"📄 Processing file {i}/{len(excel_files)}")
            print(f"   file_path: {file_path}")

            result_dict = f4_2_process_single_file(
                file_path=file_path,
                comm_month=comm_month,
                comm_tables_main_df=comm_tables_main_df,
                sh_output=sh_output,
                is_folder_run=True
            )

            processed_log.append(result_dict)

        processed_log_df = pd.DataFrame(processed_log)

        print("------------------------------------------------------------------")
        print("🟩 Folder processing complete. Processed log:")
        print(processed_log_df)

        return processed_log_df

    else:
        print("   [F4] File mode detected.")

        result_dict = f4_2_process_single_file(
            file_path=selected_path,
            comm_month=comm_month,
            comm_tables_main_df=comm_tables_main_df,
            sh_output=sh_output,
            is_folder_run=False
        )

        return pd.DataFrame([result_dict])
    
# ---------------------------------------------------------------------------
# F4.1: Get Excel files in folder
# ---------------------------------------------------------------------------
def f4_1_get_excel_files_in_folder(folder_path):
    print("   [F4.1] Creating list of Excel files in folder...")

    valid_extensions = (".xlsx", ".xls", ".xlsm", ".xlsb")

    try:
        excel_files = [
            os.path.join(folder_path, file_name)
            for file_name in os.listdir(folder_path)
            if file_name.lower().endswith(valid_extensions)
               and not file_name.startswith("~$")
        ]

        print(f"   [F4.1] Found {len(excel_files)} Excel file(s):")
        for file in excel_files:
            print(f"      - {file}")

        return excel_files

    except Exception as e:
        print(f"🔴 [F4.1] Error getting files in folder: {e}")
        print(traceback.format_exc())
        return []
    
# ---------------------------------------------------------------------------
# F4.2: Process single file
# ---------------------------------------------------------------------------
def f4_2_process_single_file(file_path, comm_month, comm_tables_main_df, sh_output, is_folder_run=False):
    print("   [F4.2] Processing single file...")
    print(f"   [F4.2] file_path: {file_path}")
    print(f"   [F4.2] is_folder_run: {is_folder_run}")

    try:
        # ---------------------------------------------------------------
        # Step 4.2.1: Try detect comm type from filename
        # ---------------------------------------------------------------
        print("   🟦 [F4.2.1] Detect comm type from filename")
        detected_comm_type = f4_2_1_detect_comm_type_label(file_path)
        print(f"   [F4.2.1] detected_comm_type: {detected_comm_type}")

        detected_reader = get_reader_for_comm_type(detected_comm_type) if detected_comm_type else None
        print(f"   [F4.2.1] detected_reader found: {detected_reader is not None}")

        # ---------------------------------------------------------------
        # Step 4.2.2: Prompt user only if detection failed / no reader
        # ---------------------------------------------------------------
        if not detected_comm_type or detected_reader is None:
            print("   🟠 [F4.2.2] No confident comm type / reader found. Prompting user...")
            comm_type = f4_2_2_prompt_comm_type_only_if_needed(detected_comm_type)
            print(f"   [F4.2.2] user selected comm_type: {comm_type}")

            if not comm_type:
                message = "No comm type selected."
                print(f"   🔴 [F4.2.2] {message}")

                if not is_folder_run:
                    f4_5_show_processing_error_widget(file_path, message)

                return {
                    "file_path": file_path,
                    "status": "error",
                    "comm_type": "",
                    "message": message
                }
        else:
            comm_type = detected_comm_type

        print(f"   ✅ [F4.2] Final comm_type to use: {comm_type}")

        # ---------------------------------------------------------------
        # Step 4.3: Match reader to comm_type
        # ---------------------------------------------------------------
        print("   🟦 [F4.3] Match reader to comm_type")
        reader_func = get_reader_for_comm_type(comm_type)
        print(f"   [F4.3] reader_func: {reader_func.__name__ if reader_func else None}")

        if reader_func is None:
            message = f"No reader found for comm_type='{comm_type}'"
            print(f"   🔴 [F4.3] {message}")

            if not is_folder_run:
                f4_5_show_processing_error_widget(file_path, message)

            return {
                "file_path": file_path,
                "status": "error",
                "comm_type": comm_type,
                "message": message
            }

        # ---------------------------------------------------------------
        # Step 4.4: Run appropriate process reader
        # ---------------------------------------------------------------
        print("   🟦 [F4.4] Run process reader")
        engineered_df = f4_4_run_engineered_reader(
            comm_type=comm_type,
            file_path=file_path,
            comm_month=comm_month,
            comm_tables_main_df=comm_tables_main_df,
            reader_func=reader_func
        )

        # ---------------------------------------------------------------
        # Step 4.5: Handle empty / error df
        # ---------------------------------------------------------------
        print("   🟦 [F4.5] Check engineered_df output")

        if engineered_df is None:
            message = "Reader returned None"
            print(f"   🔴 [F4.5] {message}")

            if not is_folder_run:
                f4_5_show_processing_error_widget(file_path, message)

            return {
                "file_path": file_path,
                "status": "error",
                "comm_type": comm_type,
                "message": message
            }

        if isinstance(engineered_df, pd.DataFrame) and engineered_df.empty:
            message = "Reader returned empty DataFrame"
            print(f"   🔴 [F4.5] {message}")

            if not is_folder_run:
                f4_5_show_processing_error_widget(file_path, message)

            return {
                "file_path": file_path,
                "status": "error",
                "comm_type": comm_type,
                "message": message
            }

        print(f"   ✅ [F4.5] Engineered DF ready. Shape: {engineered_df.shape}")

        # ---------------------------------------------------------------
        # Optional: Apply DF to output sheet
        # ---------------------------------------------------------------
        print("   🟦 [F4.5A] Apply DF to output sheet")
        apply_success, apply_message = f4_5a_apply_df_to_output(
            engineered_df=engineered_df,
            sh_output=sh_output,
            file_path=file_path,
            comm_type=comm_type
        )
        print(f"   [F4.5A] apply_success: {apply_success}")
        print(f"   [F4.5A] apply_message: {apply_message}")

        return {
            "file_path": file_path,
            "status": "success" if apply_success else "error",
            "comm_type": comm_type,
            "message": apply_message,
            "rows": len(engineered_df)
        }

    except Exception as e:
        message = f"Unhandled processing error: {e}"
        print(f"🔴 [F4.2] {message}")
        print(traceback.format_exc())

        if not is_folder_run:
            f4_5_show_processing_error_widget(file_path, message)

        return {
            "file_path": file_path,
            "status": "error",
            "comm_type": "",
            "message": message
        }
        
# ---------------------------------------------------------------------------
# F4.5A: Apply DF to output sheet
# ---------------------------------------------------------------------------
def f4_5a_apply_df_to_output(engineered_df, sh_output, file_path, comm_type):
    try:
        print("   [F4.5A] Writing df to next available row in output sheet...")

        last_used_row = sh_output.range("A" + str(sh_output.cells.last_cell.row)).end("up").row
        next_row = 2 if last_used_row < 2 else last_used_row + 1

        print(f"   [F4.5A] last_used_row: {last_used_row}")
        print(f"   [F4.5A] next_row: {next_row}")

        sh_output.range(f"A{next_row}").value = engineered_df

        return True, f"DF applied successfully for {comm_type}"

    except Exception as e:
        print(f"🔴 [F4.5A] Error applying DF: {e}")
        print(traceback.format_exc())
        return False, f"Error applying DF: {e}"
    

    
    

In [4]:
process_comms_main()

🟦 Step 1: Confirm Test / Official / File / Folder
✅ Step 1 complete
   [F1] test_official : test
   [F1] file_folder   : file
   [F1] clear_table   : None
🟦 Step 2: Get workbook, output sheet, comm_month, and comm tables df
   [F2] Getting active workbook...
   [F2] current_wb: Feb.2026.Comms_Allocator_Vs2.xlsm
   [F2] Checking output sheet...
   [F2] existing_sheet_names: ['dashboard', 'processed_comms_test', 'comm_tables', 'outputs', 'comms_merged', 'processed_comms', 'lookup_tables', 'steyn_gf_result', 'woods_steyn_result', 'woods_quin_result', 'wallace_m_result', 'steyn_nf_result']
   [F2] Using existing output sheet: processed_comms_test
   [F2] Reading comm_month from Dashboard!C3...
   [F2] ✅ Comm Month Read
   [F2] comm_month: 2026-01-01 00:00:00
   [F2] Reading lookup table from Lookup_Tables...
   [F2] Lookup_Tables sheet found
   [F2] Raw last_row from BC4000.end('up'): 45
   [F2] lookup_range: BC3:BM45
   [F2] Raw lookup values type: <class 'list'>
   [F2] ✅ comm_tables_mai